In [ ]:
import csv
from docplex.mp.model import Model

#    Reads a CSV file containing bus stop node information and returns a list of node dictionaries.
#     Each node must have three fields: 'id' (int), 'time' (int, in minutes), and 'loc' (str).
#     The function opens the file, reads each row using csv.DictReader (which maps column names to values),
#     converts 'id' and 'time' to integers, and appends each node as a dictionary to a list.

def load_nodes_from_csv(path):
    nodes = []
    with open(path, newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            node = {
                "id": int(row["id"]),
                "time": int(row["time"]),
                "loc": row["loc"]
            }
            nodes.append(node)
    return nodes

#  This function generates all valid arcs between depot and service nodes (HSK/ATB),
#  including depot-to-service, service-to-service (within 60 mins), 
# and service-to-end depot arcs, forming the edge set E for the optimization model.


def generate_arcs(V, id0, idend):
    E = []

    # Add start depot node (id0) and end depot node (idend)
    start_node = {"id": id0, "time": 0, "loc": "X"}
    end_node = {"id": idend, "time": 1440, "loc": "X"}

    # Arcs from start depot to HSK and ATB nodes starting between 1:00 (60) and 22:00 (1320)
    for v in V:
        if v["loc"] in {"HSK", "ATB"} and 60 <= v["time"] <= 1320:
            E.append({"src": start_node, "dst": v, "cost": 10, "cap": 1})

    # Group nodes by location and sort by time
    HSK_nodes = sorted([v for v in V if v["loc"] == "HSK"], key=lambda n: n["time"])
    ATB_nodes = sorted([v for v in V if v["loc"] == "ATB"], key=lambda n: n["time"])

    # Build arcs HSK -> ATB within 60 minutes difference
    for h in HSK_nodes:
        for a in ATB_nodes:
            dt = a["time"] - h["time"]
            if dt > 60:
                break
            if 0 < dt <= 60:
                E.append({"src": h, "dst": a, "cost": 10, "cap": 1})

    # Build arcs ATB -> HSK within 60 minutes difference
    for a in ATB_nodes:
        for h in HSK_nodes:
            dt = h["time"] - a["time"]
            if dt > 60:
                break
            if 0 < dt <= 60:
                E.append({"src": a, "dst": h, "cost": 10, "cap": 1})

    # Arcs from HSK and ATB nodes ending between 23:00 (1380) and 24:00 (1440) to end depot
    for v in V:
        if v["loc"] in {"HSK", "ATB"} and 1380 <= v["time"] <= 1440:
            E.append({"src": v, "dst": end_node, "cost": 10, "cap": 1})

    # Optional: add direct arc start -> end (if needed)
    # E.append({"src": start_node, "dst": end_node, "cost": 0, "cap": 1})

    return E

# --- Main model and solve ---
# This block initializes model parameters, loads nodes and arcs,
#  defines the number of vehicles and constraints, and creates decision variables for routes (x), 
# vehicle usage (z), and utilization time (T) to prepare the optimization model for solving.


def main():
    csv_path = "nodes_data.csv"
    V = load_nodes_from_csv(csv_path)

    # Define start and end depot ids
    id0 = 0      # Using 0 for start depot id (not overlapping with CSV node ids)
    idend = max(v["id"] for v in V) + 1  # end depot id just after max id

    # Generate arcs E
    E = generate_arcs(V, id0, idend)

    nVehicles = 3
    MinUtilTime = 300

    # Example frequency arrays (replace with your actual data)
    freq1 = [0]*24  # For ATB
    freq2 = [0]*24  # For HSK
    # For testing, let's say frequency of 1 at hour 1 and 2 for both locations
    freq1[1] = 1
    freq2[2] = 1

    mdl = Model("MDVSP")

    # Create variables x[src,dst,k]
    x = {}
    for k in range(nVehicles):
        for arc in E:
            key = (arc['src']['id'], arc['dst']['id'], k)
            x[key] = mdl.binary_var(name=f"x_{arc['src']['id']}{arc['dst']['id']}{k}")

    T = [mdl.integer_var(name=f"T_{k}") for k in range(nVehicles)]
    z = [mdl.binary_var(name=f"z_{k}") for k in range(nVehicles)]

    # Objective Function:

# This line minimizes the total operational cost of assigning vehicles to arcs (routes).
# Each arc has a fixed cost, and x[(src_id, dst_id, k)] is a binary variable indicating if vehicle k uses that arc.

# The expression sums the cost of all arcs used by all vehicles:
#     arc['cost'] * x[...] → cost incurred if vehicle k uses the arc.
# By summing over all arcs and vehicles, we find the lowest-cost routing plan that satisfies all constraints.
# """

    mdl.minimize(mdl.sum(arc['cost'] * x[(arc['src']['id'], arc['dst']['id'], k)]
                        for k in range(nVehicles) for arc in E))

    # MinUtilTime and linking z and x
    # For each vehicle k, this block (1) computes total travel time T[k] as the sum of costs on arcs it uses, 
    # (2) ensures z[k] is 1 if the vehicle is used (i.e., any arc assigned), and 
    # (3) enforces that used vehicles meet a minimum utilization time threshold (MinUtilTime).

    for k in range(nVehicles):
        mdl.add_constraint(
            T[k] == mdl.sum(arc['cost'] * x[(arc['src']['id'], arc['dst']['id'], k)] for arc in E)
        )
        for arc in E:
            mdl.add_constraint(z[k] >= x[(arc['src']['id'], arc['dst']['id'], k)])
        mdl.add_constraint(T[k] >= z[k] * MinUtilTime)

    node_ids = {node['id'] for node in V}
    node_ids.add(id0)
    node_ids.add(idend)

    # Flow conservation
    # This block ensures flow conservation: 
    # The start depot (id0) has exactly nVehicles outgoing arcs (one per vehicle),
    # The end depot (idend) has exactly nVehicles incoming arcs (each vehicle must finish there),
    # For all intermediate nodes and each vehicle, the number of arcs entering equals the number of arcs leaving, ensuring route continuity.

    for node_id in node_ids:
        if node_id == id0:
            mdl.add_constraint(
                mdl.sum(x[(arc['src']['id'], arc['dst']['id'], k)]
                        for k in range(nVehicles) for arc in E if arc['src']['id'] == id0) == nVehicles
            )
        elif node_id == idend:
            mdl.add_constraint(
                mdl.sum(x[(arc['src']['id'], arc['dst']['id'], k)]
                        for k in range(nVehicles) for arc in E if arc['dst']['id'] == idend) == nVehicles
            )
        else:
            for k in range(nVehicles):
                mdl.add_constraint(
                    mdl.sum(x[(arc['src']['id'], arc['dst']['id'], k)]
                            for arc in E if arc['src']['id'] == node_id) ==
                    mdl.sum(x[(arc['src']['id'], arc['dst']['id'], k)]
                            for arc in E if arc['dst']['id'] == node_id)
                )

    # Frequency constraints (per hour, per location)
    # This block enforces hourly frequency constraints:
    # For each hour (0–23), it ensures that the number of trips departing from ATB and HSK meets
    # the required minimum frequency (given by freq1 and freq2 respectively) across all vehicles.
    # Only arcs with capacity 1 and that start in the specific hour and location are considered.

    for h in range(24):
        mdl.add_constraint(
            mdl.sum(x[(arc['src']['id'], arc['dst']['id'], k)]
                    for k in range(nVehicles) for arc in E
                    if arc['cap'] == 1 and arc['src']['loc'] == "ATB" and arc['src']['time'] // 60 == h)
            >= freq1[h]
        )
        mdl.add_constraint(
            mdl.sum(x[(arc['src']['id'], arc['dst']['id'], k)]
                    for k in range(nVehicles) for arc in E
                    if arc['cap'] == 1 and arc['src']['loc'] == "HSK" and arc['src']['time'] // 60 == h)
            >= freq2[h]
        )

    # Capacity constraints
     # Ensures that the total number of vehicles using any arc (from src to dst) does not exceed its capacity.
     # For each arc, it sums usage across all vehicles and restricts it to be ≤ arc['cap'], preserving network feasibility.

    for arc in E:
        mdl.add_constraint(
            mdl.sum(x[(arc['src']['id'], arc['dst']['id'], k)] for k in range(nVehicles)) <= arc['cap']
        )

    # Solve
    # Solves the optimization model and prints the objective value (total cost). Then, for each vehicle, it traces and prints
    # the arcs (routes) used in the solution based on decision variable values, showing how each vehicle moves through the network.

    solution = mdl.solve(log_output=True)
    if not solution:
        print("No solution found")
        return

    print(f"Objective value: {solution.objective_value}\n")

    for k in range(nVehicles):
        print(f"Route for vehicle {k+1}:")
        for arc in E:
            val = x[(arc['src']['id'], arc['dst']['id'], k)].solution_value
            if val > 0.5:
                src = arc['src']
                dst = arc['dst']
                print(f"  From {src['loc']}({src['id']}, time={src['time']}) -> "
                      f"{dst['loc']}({dst['id']}, time={dst['time']})")

if _name_ == "_main_":
    main()